***

*Course:* [Math 535](https://people.math.wisc.edu/~roch/mmids/) - Mathematical Methods in Data Science (MMiDS)  
*Chapter:* 2-Least squares   
*Author:* [Sebastien Roch](https://people.math.wisc.edu/~roch/), Department of Mathematics, University of Wisconsin-Madison  
*Updated:* Jan 4, 2024   
*Copyright:* &copy; 2024 Sebastien Roch

***

In [ ]:
# IF RUNNING ON GOOGLE COLAB, UNCOMMENT THE FOLLOWING CODE CELL
# When prompted, upload: 
#     * mmids.py
#     * advertising.csv 
# from your local file system
# Files at: https://github.com/MMiDS-textbook/MMiDS-textbook.github.io/tree/main/utils
# Alternative instructions: https://colab.research.google.com/notebooks/io.ipynb

In [ ]:
#from google.colab import files
#
#uploaded = files.upload()
#
#for fn in uploaded.keys():
#    print('User uploaded file "{name}" with length {length} bytes'.format(
#      name=fn, length=len(uploaded[fn])))

In [ ]:
# FOR BOOK ONLY
import warnings
warnings.filterwarnings('ignore')
import os, sys
sys.path.insert(0, os.path.abspath('../../utils')) # use directory to mmids.py

In [ ]:
# PYTHON 3
import numpy as np
from numpy import linalg as LA
from numpy.random import default_rng
rng = default_rng(535)
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import mmids

In [ ]:
# FOR TeX ONLY
#plt.rcParams['figure.figsize'] = [4.00, 2.25] # for high-def figs
#plt.rcParams['figure.dpi'] = 200 # for high-def figs

## Background: linear spaces

In this section, we introduce some basic linear algebra concepts that will be needed throughout this chapter (and later on).

### Subspaces

Throughout, we work over the vector space $V = \mathbb{R}^n$.  We will use the notation $[m] = \{1,\ldots,m\}$. 

We begin with the concept of a linear subspace.

**DEFINITION** **(Linear Subspace)** A linear subspace of $\mathbb{R}^n$ is a subset $U \subseteq \mathbb{R}^n$ that is closed under vector addition and scalar multiplication. That is, for all $\mathbf{u}_1, \mathbf{u}_2 \in U$ and $\alpha \in \mathbb{R}$, it holds that

$$
\mathbf{u}_1 + \mathbf{u}_2 \in U \quad \text{and} \quad \alpha \,\mathbf{u}_1 \in U.
$$

It follows from this condition that $\mathbf{0} \in U$. $\natural$

Alternatively, we can check these conditions by proving that (1) $\mathbf{0} \in U$ and (2) $\mathbf{u}_1, \mathbf{u}_2 \in U$ and $\alpha \in \mathbb{R}$ imply that $\alpha \mathbf{u}_1 + \mathbf{u}_2 \in U$. Indeed, taking $\alpha = 1$ gives the first condition above, while choosing $\mathbf{u}_2 = \mathbf{0}$ gives the second one. 

**NUMERICAL CORNER:** The plane $P$ made of all points $(x,y,z) \in \mathbb{R}^3$ that satisfy $z = x+y$ is a linear subspace. Indeed, $0 = 0 + 0$ so $(0,0,0) \in P$. And, for any $\mathbf{u}_1 = (x_1, y_1, z_1)$ and $\mathbf{u}_2 = (x_2, y_2, z_2)$ such that $z_1 = x_1 + y_1$ and $z_2 = x_2 + y_2$ and for any $\alpha \in \mathbb{R}$, we have

$$
\alpha z_1 + z_2 = \alpha (x_1 + y_1) + (x_2 + y_2) = (\alpha x_1 + x_2) + (\alpha y_1 + y_2).
$$

That is, $\alpha \mathbf{u}_1 + \mathbf{u}_2$ satisfies the condition defining $P$ and therefore is itself in $P$. Note also that $P$ passes through the origin.

In this example, the linear subspace $P$ can be described alternatively as the collection of every vector of the form $(x, y, x+y)$.

We use [`plot_surface`](https://matplotlib.org/stable/api/_as_gen/mpl_toolkits.mplot3d.axes3d.Axes3D.html#mpl_toolkits.mplot3d.axes3d.Axes3D.plot_surface) to plot it over a grid of points created using [`numpy.meshgrid`](https://numpy.org/doc/stable/reference/generated/numpy.meshgrid.html).

In [ ]:
x = np.linspace(0,1,num=101)
y = np.linspace(0,1,num=101)
X, Y = np.meshgrid(x, y)

In [ ]:
print(X)

In [ ]:
print(Y)

In [ ]:
Z = X + Y
print(Z)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, Z)
plt.show()

$\unlhd$

Here is a key example of a linear subspace.

**DEFINITION** **(Span)** Let $\mathbf{w}_1, \ldots, \mathbf{w}_m \in \mathbb{R}^n$. The span of $\{\mathbf{w}_1, \ldots, \mathbf{w}_m\}$, denoted $\mathrm{span}(\mathbf{w}_1, \ldots, \mathbf{w}_m)$, is the set of all linear combinations of the $\mathbf{w}_j$'s. That is,

$$
\mathrm{span}(\mathbf{w}_1, \ldots, \mathbf{w}_m)
= \left\{
\sum_{j=1}^m \alpha_j \mathbf{w}_j\,:\, \alpha_1,\ldots, \alpha_m \in \mathbb{R}
\right\}.
$$

By convention, we declare that the span of the empty list is $\{\mathbf{0}\}$. $\natural$

**EXAMPLE:** **(continued)** In the example above, we noted that the plane $P$ is the collection of every vector of the form $(x, y, x+y)^T$. These can be written as $x \,\mathbf{w}_1 + y \,\mathbf{w}_2$ where $\mathbf{w}_1 = (1,0,1)^T$ and $\mathbf{w}_2 = (0,1,1)^T$, and vice versa. Hence $P = \mathrm{span}(\mathbf{w}_1,\mathbf{w}_2)$. $\lhd$

We check next that a span is indeed a linear subspace:

**LEMMA** Let $W=\mathrm{span}(\mathbf{w}_1, \ldots, \mathbf{w}_m)$. Then $W$ is a linear subspace. $\flat$

*Proof:* First, $\mathbf{0} = \sum_{j=1}^m 0\mathbf{w}_j \in W$. Second, let $\mathbf{u}_1, \mathbf{u}_2 \in W$ and $\alpha \in \mathbb{R}$. For $i=1,2$, because $\mathbf{u}_i$ is in the span of the $\mathbf{w}_j$'s, we can write

$$
\mathbf{u}_i = \sum_{j=1}^m \beta_{ij} \mathbf{w}_j
$$

for some $\beta_{ij} \in \mathbb{R}$ for $j=1,\ldots,m$.

Therefore

$$
\alpha \mathbf{u}_1 + \mathbf{u}_2 
= \alpha \sum_{j=1}^m \beta_{1j} \mathbf{w}_j
+ \sum_{j=1}^m \beta_{2j} \mathbf{w}_j
= \sum_{j=1}^m (\alpha \beta_{1j} + \beta_{2j}) \mathbf{w}_j.
$$

So $\alpha \,\mathbf{u}_1 + \mathbf{u}_2 \in W$.$\square$

In matrix form, we talk about the column space of a (not necessarily square) matrix.

**DEFINITION** **(Column Space)** Let $A \in \mathbb{R}^{n\times m}$ be an $n\times m$ matrix with columns $\mathbf{a}_1,\ldots, \mathbf{a}_m \in \mathbb{R}^n$. The column space of $A$, denoted $\mathrm{col}(A)$, is the span of the columns of $A$, that is, $\mathrm{col}(A) = \mathrm{span}(\mathbf{a}_1,\ldots, \mathbf{a}_m)$. $\natural$

(When thinking of $A$ as a linear map, the column space is often referred to as the range or image.)

We will need another important linear subspace defined in terms of a matrix. This one is defined implicitly in terms of the given matrix.

**DEFINITION** **(Null Space)** Let $B \in \mathbb{R}^{n \times m}$. The null space of $B$ is the linear subspace

$$
\mathrm{null}(B)
= \left\{\mathbf{x} \in \mathbb{R}^m\,:\, B\mathbf{x} = \mathbf{0}\right\}.
$$

$\natural$

It can be shown that the null space is a linear subspace. We give a simple example next.

**EXAMPLE:** **(continued)** Going back to the linear subspace $P = \{(x,y,z)^T \in \mathbb{R}^3 : z = x + y\}$,  the condition in the definition can be re-written as $x + y - z = 0$. Hence $P = \mathrm{null}(B)$ for the single-row matrix $B = \begin{pmatrix} 1 & 1 & - 1\end{pmatrix}$. $\lhd$

### Linear independence

It is often desirable to avoid redundancy in the description of a linear subspace. 

We start with an example.

**EXAMPLE:** Consider the linear subspace $\mathrm{span}(\mathbf{w}_1,\mathbf{w}_2,\mathbf{w}_3)$, where $\mathbf{w}_1 = (1,0,1)^T$, $\mathbf{w}_2 = (0,1,1)^T$, and $\mathbf{w}_3 = (1,-1,0)^T$. We claim that 

$$
\mathrm{span}(\mathbf{w}_1,\mathbf{w}_2,\mathbf{w}_3) = \mathrm{span}(\mathbf{w}_1,\mathbf{w_2}).
$$

Recall that to prove an equality between sets, it suffices to prove inclusion in both directions. 

First, it is immediate by definition of the span that 

$$
\mathrm{span}(\mathbf{w}_1,\mathbf{w}_2) \subseteq \mathrm{span}(\mathbf{w}_1,\mathbf{w}_2,\mathbf{w}_3).
$$

To prove the other direction, let $\mathbf{u} 
\in \mathrm{span}(\mathbf{w}_1,\mathbf{w}_2,\mathbf{w}_3)$ so that

$$
\mathbf{u}
= \beta_1\,(1,0,1)^T + \beta_2\,(0,1,1)^T + \beta_3\,(1,-1,0)^T.
$$

Now observe that $(1,-1,0)^T = (1,0,1)^T - (0,1,1)^T$. Put differently, $\mathbf{w}_3 \in \mathrm{span}(\mathbf{w}_1,\mathbf{w}_2)$. Replacing above gives

$$
\begin{align*}
\mathbf{u}
&= \beta_1\,(1,0,1)^T + \beta_2\,(0,1,1)^T + \beta_3\,[(1,0,1)^T - (0,1,1)^T]\\
&= (\beta_1+\beta_3)\,(1,0,1)^T + (\beta_2-\beta_3)\,(0,1,1)^T
\end{align*}
$$

which shows that $\mathbf{u} \in \mathrm{span}(\mathbf{w}_1,\mathbf{w_2})$.
In words, $(1,-1,0)^T$ is redundant.
Hence 

$$
\mathrm{span}(\mathbf{w}_1,\mathbf{w}_2) \supseteq \mathrm{span}(\mathbf{w}_1,\mathbf{w}_2,\mathbf{w}_3)
$$ 

and that concludes the proof.$\lhd$

**DEFINITION** **(Linear Independence)** A list of nonzero vectors $\mathbf{u}_1,\ldots,\mathbf{u}_m$ is linearly independent if none of them can be written as a linear combination of the others, that is,

$$
\forall i,\ \mathbf{u}_i \notin \mathrm{span}(\{\mathbf{u}_j:j\neq i\}).
$$

By convention, we declare the empty list to be linearly independent. A list of vectors is called linearly dependent if it is not linearly independent. $\natural$

**EXAMPLE:** **(continued)** In the previous example, $\mathbf{w}_1,\mathbf{w}_2,\mathbf{w}_3$ are *not* linearly independent, because we showed that $\mathbf{w}_3$ can be written as a linear combination of $\mathbf{w}_1,\mathbf{w}_2$. On the other hand, $\mathbf{w}_1,\mathbf{w}_2$ are linearly independent because there is no $\alpha, \beta  \in \mathbb{R}$ such that $(1,0,1)^T = \alpha\,(0,1,1)^T$ or $(0,1,1)^T = \beta\,(1,0,1)^T$. Indeed, the first equation requires $1 = \alpha \, 0$ (first component) and the second one requires $1 = \beta \, 0$ (second component) - both of which have no solution.
$\lhd$

**LEMMA** **(Equivalent Definition of Linear Independence)** Vectors $\mathbf{u}_1,\ldots,\mathbf{u}_m$ are linearly independent if and only if

$$
\sum_{j=1}^m \alpha_j \mathbf{u}_j = \mathbf{0} \implies \alpha_j = 0,\ \forall j.
$$

Equivalently, $\mathbf{u}_1,\ldots,\mathbf{u}_m$ are linearly dependent if and only if there exist $\alpha_j$'s, not all zero, such that $\sum_{j=1}^m \alpha_j \mathbf{u}_j = \mathbf{0}$. 

$\flat$

*Proof:* We prove the second statement.

- Assume $\mathbf{u}_1,\ldots,\mathbf{u}_m$ are linearly dependent. Then $\mathbf{u}_i = \sum_{j\neq i} \alpha_j \mathbf{u}_j$ for some $i$. Taking $\alpha_i = -1$ gives $\sum_{j=1}^m \alpha_j \mathbf{u}_j = \mathbf{0}$.

- Assume $\sum_{j=1}^m \alpha_j \mathbf{u}_j = \mathbf{0}$ with $\alpha_j$'s not all zero. In particular $\alpha_i \neq 0$ for some $i$. Then 

$$
\mathbf{u}_i 
= - \frac{1}{\alpha_i} \sum_{j\neq i} \alpha_j \mathbf{u}_j
= \sum_{j\neq i} \left(- \frac{\alpha_j}{\alpha_i}\right) \mathbf{u}_j.
$$

$\square$ 

In matrix form: let $\mathbf{a}_1,\ldots,\mathbf{a}_m \in \mathbb{R}^n$ and form the matrix whose columns are the $\mathbf{a}_i$'s

$$
A =
\begin{pmatrix}
| &  & | \\
\mathbf{a}_1 & \ldots & \mathbf{a}_m \\
| &  & | 
\end{pmatrix}.
$$

Note that $A\mathbf{x}$ is the following linear combination of the columns of $A$: $\sum_{j=1}^m x_j \mathbf{a}_j$. Hence $\mathbf{a}_1,\ldots,\mathbf{a}_m$ are linearly independent if and only if
$A \mathbf{x} = \mathbf{0} \implies \mathbf{x} = \mathbf{0}$. In terms of the null space of $A$, this last condition translates into $\mathrm{null}(A) = \{\mathbf{0}\}$.

Equivalently, $\mathbf{a}_1,\ldots,\mathbf{a}_m$ are linearly dependent if and only if $\exists \mathbf{x}\neq \mathbf{0}$ such that $A \mathbf{x} = \mathbf{0}$. Put differently, this last condition means that there is a nonzero vector in the null space of $A$.

In this course, we will *not* be interested in checking these types of conditions by hand (although you may have learned to do so in a prior course).

Bases give a convenient - non-redundant - representation of a subspace.

**DEFINITION** **(Basis)** Let $U$ be a linear subspace of $\mathbb{R}^n$. A basis of $U$ is a list of vectors $\mathbf{u}_1,\ldots,\mathbf{u}_m$ in $U$ that: (1) span $U$, that is, $U = \mathrm{span}(\mathbf{u}_1,\ldots,\mathbf{u}_m)$; and (2) are linearly independent. $\natural$

We denote by $\mathbf{e}_1, \ldots, \mathbf{e}_n$ the standard basis of $\mathbb{R}^n$, where $\mathbf{e}_i$ has a one in coordinate $i$ and zeros in all other coordinates (see the example below). The basis of the linear subspace $\{\mathbf{0}\}$ is the empty list (which, by convention, is independent and has span $\{\mathbf{0}\}$).

**EXAMPLE:** **(continued)** In the previous example, the vectors $\mathbf{w}_1,\mathbf{w}_2$ are a basis of their span $U = \mathrm{span}(\mathbf{w}_1,\mathbf{w}_2)$. Indeed the first condition is trivially satisfied. Plus, we have shown above that $\mathbf{w}_1,\mathbf{w}_2$ are linearly independent. $\lhd$

**EXAMPLE:** For $i=1,\ldots,n$, define $\mathbf{e}_i \in \mathbb{R}^n$ as the vector with entries

$$
(\mathbf{e}_i)_j 
= \begin{cases}
1, & \text{if $j = i$,}\\
0, & \text{o.w.}
\end{cases}
$$

Then $\mathbf{e}_1, \ldots, \mathbf{e}_n$ form a basis of $\mathbb{R}^n$, as each vector is in $\mathbb{R}^n$. It is known as the standard basis of $\mathbb{R}^n$. Indeed, clearly $\mathrm{span}(\mathbf{e}_1, \ldots, \mathbf{e}_n) \subseteq \mathbb{R}^n$. Moreover, any vector $\mathbf{u} = (u_1,\ldots,u_n)^T \in \mathbb{R}^n$ can be written as

$$
\mathbf{u}
= \sum_{i=1}^{n} u_i \mathbf{e}_i.
$$

So $\mathbf{e}_1, \ldots, \mathbf{e}_n$ spans $\mathbb{R}^n$. Furthermore, 

$$
\mathbf{e}_i \notin \mathrm{span}(\{\mathbf{e}_j:j\neq i\}), \quad \forall i=1,\ldots,n,
$$

since $\mathbf{e}_i$ has a non-zero $i$-th entry while all vectors on the right-hand side have a zero in entry $i$. So the vectors $\mathbf{e}_1, \ldots, \mathbf{e}_n$ are linearly independent. $\lhd$

A key property of a basis is that it provides a *unique* representation of the vectors in the subspace. Indeed, let $U$ be a linear subspace and $\mathbf{u}_1,\ldots,\mathbf{u}_m$ be a basis of $U$. Suppose that $\mathbf{w} \in U$ can be written as both $\mathbf{w} = \sum_{j=1}^m \alpha_j \mathbf{u}_j$ and $\mathbf{w} = \sum_{j=1}^m \alpha_j' \mathbf{u}_j$. Then subtracting one equation from the other we arrive at $\sum_{j=1}^m (\alpha_j - \alpha_j') \,\mathbf{u}_j = \mathbf{0}$. By linear independence, we have $\alpha_j - \alpha_j' = 0$ for each $j$. That is, there is only one way to express $\mathbf{w}$ as a linear combination of the basis. 

The basis itself on the other hand is not unique. 

### Dimension of a subspace

A second key property of a basis is that it always has the *same number of elements*, which is called the dimension of the subspace.

**THEOREM** **(Dimension)** Let $U \neq \{\mathbf{0}\}$ be a linear subspace of $\mathbb{R}^n$. Then all bases of $U$ have the same number of elements. We call this number the dimension of $U$ and denote it by $\mathrm{dim}(U)$. $\sharp$

The proof is optional and is provided in the next section. It relies on the *Linear Dependence Lemma*, which is also stated in the next section. That fundamental lemma has many useful implications, some of which we state now.

 A list of linearly independent vectors in a subspace $U$ is referred to as an independent list in $U$. A list of vectors whose span is $U$ is referred to as a spanning list of $U$. In the lemmas below, $U$ is a linear subspace of $\mathbb{R}^n$. The first and second lemmas are proved in a later section. The *Dimension Theorem* immediately follows from the first one (Why?).

**LEMMA** **(Independent is Shorter than Spanning)**  The length of any independent list in $U$ is less or equal than the length of any spanning list of $U$. $\flat$


**LEMMA** **(Completing an Independent List)** Any independent list in $U$ can be completed into a basis of $U$. $\flat$

**LEMMA** **(Reducing a Spanning List)** Any spanning list of $U$ can be reduced into a basis of $U$. $\flat$

We mention a few observations implied by the previous lemmas. 

**Observation D1:** Any linear subspace $U$ of $\mathbb{R}^n$ has a basis. To show this, start with the empty list and use the *Completing an Independent List Lemma* to complete it into a basis of $U$. Observe further that, instead of an empty list, we could have initialized the process with a list containing any vector in $U$. That is, for any non-zero vector $\mathbf{u} \in U$, we can construct a basis of $U$ that includes $\mathbf{u}$.

**Observation D2:** The dimension of any linear subspace $U$ of $\mathbb{R}^n$ is smaller or equal than $n$. Indeed, because a basis of $U$ is an independent list in the full space $\mathbb{R}^n$, by the *Completing an Independent List Lemma* it can be completed into a basis of $\mathbb{R}^n$, which has $n$ elements by *Dimension Theorem* (and the fact that the standard basis has $n$ elements). A similar statement holds more generally for nested linear subspaces $U \subseteq V$, that is, $\mathrm{dim}(U) \leq \mathrm{dim}(V)$.

When applied to a matrix $A$, the dimension of the column space of $A$ is called the column rank of $A$. Similarly the row rank of $A$ is the dimension of its row space.

**DEFINITION** **(Row Space)** The row space of $A \in \mathbb{R}^{n \times m}$, denoted $\mathrm{row}(A)$, is the span of the rows of $A$ as vectors in $\mathbb{R}^m$. $\natural$

Observe that the row space of $A$ is equal to the column space of its transpose $A^T$.

As it turns out, these two notions of rank are the same. From now on, we refer to the row rank and column rank of $A$ simply as the rank, which we denote by $\mathrm{rk}(A)$.

**THEOREM** **(Row Rank Equals Column Rank)** For any $A \in \mathbb{R}^{n \times m}$, the row rank of $A$ equals the column rank of $A$. Moreover, $\mathrm{rk}(A) \leq \min\{n,m\}$. $\sharp$

*Proof idea:* Write $A$ as a matrix factorization $BC$ where the columns of $B$ form a basis of $\mathrm{col}(A)$. Then the rows of $C$ necessarily form a spanning set of $\mathrm{row}(A)$. So, because the number of columns of $B$ and the number of rows of $C$ match, we conclude that the row rank is less or equal than the column rank. Applying the same argument to the transpose gives the claim.

*Proof:* Assume that $A$ has column rank $r$. Then there exists a basis $\mathbf{b}_1,\ldots, \mathbf{b}_r \in \mathbb{R}^n$ of $\mathrm{col}(A)$ by *Observation D1* above, and we know that $r \leq n$ by *Observation D2*. That is, for each $j$, letting $\mathbf{a}_{j} = A_{\cdot,j}$ be the $j$-th column of $A$ we can write

$$
\mathbf{a}_{j} = \sum_{\ell=1}^r \mathbf{b}_\ell c_{\ell j}
$$

for some $c_{\ell j}$'s. Let $B$ be the matrix whose columns are $\mathbf{b}_1, \ldots, \mathbf{b}_r$ and let $C$ be the matrix with entries $(C)_{\ell j} = c_{\ell j}$, $\ell=1,\ldots,r$, $j=1,\ldots,m$. Then the equation above can be re-written as the matrix factorization $A = BC$. Indeed, by our previous observations about matrix-matrix products, the columns of $A$ are linear combinations of the columns of $B$ with coefficients taken from the corresponding column of $C$.

The key point is the following: $C$ necessarily has $r$ rows. Let $\boldsymbol{\alpha}_{i}^T = A_{i,\cdot}$ be the $i$-th row of $A$ and $\mathbf{c}_{\ell}^T = C_{\ell,\cdot}$ be the $\ell$-th row of $C$. Using our alternative representation of matrix-matrix product in terms of rows, the decomposition is equivalent to

$$
\boldsymbol{\alpha}_{i}^T = \sum_{\ell=1}^r b_{i\ell} \mathbf{c}_\ell^T, \quad i=1,\ldots, n,
$$

where $b_{i\ell} = (\mathbf{b}_i)_\ell = (B)_{i\ell}$ is the $i$-th entry of the $\ell$-th column of $B$. In words, the rows of $A$ are linear combinations of the rows of $C$ with coefficients taken from the corresponding row of $B$. In particular, $\mathcal{C} = \{\mathbf{c}_{j}:j=1,\ldots,r\}$ is a spanning list of the row space of $A$, that is, each row of $A$ can be written as a linear combination of $\mathcal{C}$. 

So the row rank of $A$ is at most $r$, the column rank of $A$, by *Observation D2*.

Applying the same argument to $A^T$, which switches the role of the columns and the rows, gives that the column rank of $A$ (i.e. the row rank of $A^T$) is at most the row rank of $A$ (i.e. the column rank of $A^T$). Hence the two notions of rank must be equal. (We also deduce $r \leq m$ by *Observation D2* again.) $\square$

**EXAMPLE:** **(continued)** We illustrate the proof of the theorem. Continuing a previous example, let $A$ be the matrix with columns $\mathbf{w}_1 = (1,0,1)^T$, $\mathbf{w}_2 = (0,1,1)^T$, and $\mathbf{w}_3 = (1,-1,0)^T$

$$
A
= 
\begin{pmatrix}
1 & 0 & 1\\
0 & 1 & -1\\
1 & 1 & 0
\end{pmatrix}.
$$

We know that $\mathbf{w}_1$ and $\mathbf{w}_2$ form a basis of $\mathrm{col}(A)$. We use them to construct our matrix $B$

$$
B
= 
\begin{pmatrix}
1 & 0\\
0 & 1\\
1 & 1
\end{pmatrix}.
$$

Recall that $\mathbf{w}_3 = \mathbf{w}_1 - \mathbf{w}_2$. Hence the matrix $C$ is

$$
C
= 
\begin{pmatrix}
1 & 0 & 1\\
0 & 1 & -1
\end{pmatrix}.
$$

Indeed, column $j$ of $C$ gives the coefficients in the linear combination of the columns of $B$ that produces column $j$ of A. Check that $A = B C$. $\lhd$

**NUMERICAL CORNER:** In Numpy, one can compute the rank of a matrix using the function [`numpy.linalg.matrix_rank`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.matrix_rank.html). We will see later in the course how to compute it using the singular value decomposition (which is how `LA.matrix_rank` does it).

Let's try the example above.

In [ ]:
w1 = np.array([1., 0., 1.])
w2 = np.array([0., 1., 1.])
w3 = np.array([1., -1., 0.])
A = np.stack((w1, w2, w3), axis=-1)
print(A)

We compute the rank of `A`.

In [ ]:
LA.matrix_rank(A)

We take only the first two columns of `A` this time to form `B`.

In [ ]:
B = np.stack((w1, w2),axis=-1)
print(B)

In [ ]:
LA.matrix_rank(B)

In Numpy, `@` is used for matrix product.

In [ ]:
C = np.array([[1., 0., 1.],[0., 1., -1.]])
print(C)

In [ ]:
LA.matrix_rank(C)

In [ ]:
print(B @ C)

$\unlhd$

**EXAMPLE:** Let $A \in \mathbb{R}^{n \times k}$ and $B \in \mathbb{R}^{k \times m}$. Then we claim that

$$
\mathrm{rk}(AB) \leq \mathrm{rk}(A).
$$

Indeed, the columns of $AB$ are linear combinations of the columns of $A$. Hence $\mathrm{col}(AB) \subseteq \mathrm{col}(A)$. Because a basis of $\mathrm{col}(AB)$ is an independent list in $\mathrm{col}(A)$, it can be completed into a basis of $\mathrm{col}(A)$ by the *Completing an Independent List Lemma*. The length of the resulting basis is greater or equal than the length of the basis of $\mathrm{col}(AB)$. Because the rank of a matrix is the length of a basis of its column space, the claim follows. $\lhd$

**EXAMPLE:** Let $A \in \mathbb{R}^{n \times k}$ and $B \in \mathbb{R}^{k \times m}$. Then we claim that

$$
\mathrm{rk}(A + B) \leq \mathrm{rk}(A) + \mathrm{rk}(B).
$$

Indeed, the columns of $A + B$ are linear combinations of the columns of $A$ and of $B$. Let $\mathbf{u}_1,\ldots,\mathbf{u}_h$ be a basis of $\mathrm{col}(A)$ and let $\mathbf{v}_1,\ldots,\mathbf{v}_{\ell}$ be a basis of $\mathrm{col}(B)$. Then, we deduce 

$$
\mathrm{col}(A + B) \subseteq \mathrm{span}(\mathbf{u}_1,\ldots,\mathbf{u}_h,\mathbf{v}_1,\ldots,\mathbf{v}_{\ell}).
$$ 

Using the *Completing an Independent List Lemma* as in the previous example, it follows that

$$
\mathrm{rk}(A + B) \leq \mathrm{dim}(\mathrm{span}(\mathbf{u}_1,\ldots,\mathbf{u}_h,\mathbf{v}_1,\ldots,\mathbf{v}_{\ell})).
$$ 

By the *Reducing a Spanning List Lemma*, the right hand side is at most the length of the spanning list, i.e., $h+\ell$. But by construction $\mathrm{rk}(A) = h$ and $\mathrm{rk}(B) = \ell$, so we are done. $\lhd$